# 第 1 天实验室 —— 你的第一个前沿 LLM 项目

## 练习目标

给一个 URL，抓取网页正文，再调用 Frontier 模型生成**简短摘要**（可带一点幽默语气）。这是整门课的起点：后面会一路走到多 Agent 协作。

## 和本课概念的对照

| 本课概念 | 本笔记本里你会看到 |
|----------|------------------|
| 环境变量 / `.env` | `load_dotenv` + `OPENAI_API_KEY` 校验 |
| Chat Completions | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定语气与任务，user 塞网页正文 |
| 网页抓取 | `scraper.fetch_website_contents` |
| IPython 展示 | `display(Markdown(...))` |

## 怎么跑

1. 读课程 [README.md](../README.md) 与 [资源页](https://edwarddonner.com/2024/11/13/llm-engineering-resources/)
2. 选中内核：推荐 `.venv (Python 3.12.x)`
3. 准备 `.env` 中的 `OPENAI_API_KEY`（以 `sk-proj-` 开头）
4. **从上到下**依次 Shift+Enter 运行每个单元格
5. 指南见 [Guides](../guides/01_intro.ipynb)；排错见 [troubleshooting](../setup/troubleshooting.ipynb)

## 学习建议

- 先看讲座，再自己跑通本笔记本；多加 `print`，再改 URL / 提示词做变体
- 有 GitHub 可展示你的变体（对求职也有帮助）
- 代码会随课程更新；Udemy「公告」里有作者推送
- 问题可联系：平台私信 / ed@edwarddonner.com / [LinkedIn](https://www.linkedin.com/in/eddonner/) / [@edwarddonner](https://x.com/edwarddonner)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">请阅读 — 重要说明</h2>
            <span style="color:#900;">作者偏好用 Jupyter Lab 演示而不是对着屏幕打字。建议你在<b>看完讲座之后</b>自己仔细跑一遍：加打印、改变体。若有 GitHub，用它展示你的版本——既是练习，也能给未来雇主看。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">此代码是实时资源</h2>
            <span style="color:#f71;">作者会定期推送更新（更多例子、更好注释、新模型如 DeepSeek）。笔记本内容可能与视频略有差别，但视频里的要点都在这里。请留意 Udemy 公告/邮件。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业价值</h2>
            <span style="color:#181;">练习可以好玩（讲笑话、互相抬杠），但核心是可落地的商业技能：摘要可用于新闻、财报、简历/求职信等。边做边想：这能用在你的业务里吗？</span>
        </td>
    </tr>
</table>


### 若需要：安装 Cursor / VS Code 扩展

1. 菜单 **View → Extensions**
2. 搜索 **Python**，安装 Microsoft（`ms-python`）的 Python 扩展
3. 搜索 **Jupyter**，安装 Microsoft（`ms-toolsai`）的 Jupyter 扩展

### 接着选择内核（Kernel）

1. 点右上角 **Select Kernel**
2. 选 **Python Environments...**
3. 选带推荐标记的 `.venv (Python 3.12.x) .venv/bin/python`

有问题？去故障排除笔记本。

**注意：** 每个笔记本都要单独选一次内核。


In [ ]:
# ========== 导入：本实验要用的库 ==========

# 导入标准库 os：读环境变量（例如 OPENAI_API_KEY）
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件载入环境变量
from dotenv import load_dotenv
# 从本地 scraper 导入：按 URL 抓取网页可见文本
from scraper import fetch_website_contents
# 从 IPython.display 导入：在笔记本里渲染 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用 Chat Completions API
from openai import OpenAI

# 若本格报错（缺包/路径问题），先去故障排除笔记本排查


# 连接到 OpenAI（或 Ollama）

下一格会从 `.env` 加载环境变量并检查 `OPENAI_API_KEY`。

若想用免费的 **Ollama**，见 README「付费 API 的免费替代方案」；完整示例可参考解决方案里的 `day1_with_ollama.ipynb`。

## 排错提示

- **NameError**：是否从上到下跑过所有格子？见 Python 基础指南
- 仍不行：打开 [troubleshooting](../setup/troubleshooting.ipynb)
- 或联系作者 ed@edwarddonner.com
- API 费用：README 有说明；也可用 Ollama（Day 2 会展开）


In [ ]:
# ========== 加载 .env 并校验 OPENAI_API_KEY ==========

# override=True：用 .env 覆盖已存在的同名环境变量
load_dotenv(override=True)
# 读取 OpenAI 密钥
api_key = os.getenv('OPENAI_API_KEY')

# 分档检查：缺失 / 前缀不对 / 首尾空白 —— 打印英文提示便于对照官方排错文案
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


# 热身：先对 Frontier 模型发一条最短消息

在搭「网页摘要」之前，先确认 API 通路是通的。


In [ ]:
# ========== 热身 1：构造 messages（还不发请求） ==========

# 发给模型的用户文本（保持英文原文）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# Chat Completions 期望的列表结构：每项含 role + content
messages = [{"role": "user", "content": message}]

# 笔记本里最后一行表达式会展示 messages，便于确认结构
messages


In [ ]:
# ========== 热身 2：真正调用 OpenAI Chat Completions ==========

# 使用默认环境里的 OPENAI_API_KEY 创建客户端
openai = OpenAI()

# model 与 messages 决定本次调用；返回值里含 choices
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# 取出第一条助手回复的文本内容（笔记本会显示该表达式结果）
response.choices[0].message.content


## 开始第一个项目：URL → 网页正文 → 摘要


In [ ]:
# ========== 试抓取：用 scraper 取一个网站正文 ==========

# 目标网站 URL（可改成你想摘要的站点）
url = "https://www.nafnaf.com.co"
# 调用课程提供的抓取工具，返回清洗后的文本
website = fetch_website_contents(url)
# 打印正文，确认抓取是否成功
print(website)


## 提示类型（Prompt Types）

Frontier 模型通常按角色接收指令：

- **System prompt**：任务是什么、语气如何
- **User prompt**：对话起点 / 要处理的具体内容（这里是网页正文）


In [ ]:
# ========== 定义 system prompt：尖酸幽默的网站摘要助手 ==========
# 可自行改最后一句语气/语言；但发给模型的字符串保持英文可运行原样

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [ ]:
# ========== 定义 user prompt 前缀：后面会拼上网站正文 ==========

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## Messages 结构

OpenAI API（以及很多兼容 API）期望类似：

```python
[
    {"role": "system", "content": "系统消息"},
    {"role": "user", "content": "用户消息"}
]
```

下面几格先用「2+2」小例子感受 **system 如何改变语气**，再接到网页摘要。


In [ ]:
# ========== 语气实验：system =「尖酸」时，同一道 2+2 会怎样答 ==========

# 构造仅含 system + user 的 messages；user 问题保持英文
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小的 gpt-4.1-nano 快速看 system 对风格的影响
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 展示助手回复文本
response.choices[0].message.content


In [ ]:
# ========== 语气实验：system =「哥伦比亚麦德林」时，同一道 2+2 会怎样答 ==========

# 构造仅含 system + user 的 messages；user 问题保持英文
messages = [
    {"role": "system", "content": "You are a colombian from medellin assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小的 gpt-4.1-nano 快速看 system 对风格的影响
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 展示助手回复文本
response.choices[0].message.content


In [ ]:
# ========== 语气实验：system =「粗鲁」时，同一道 2+2 会怎样答 ==========

# 构造仅含 system + user 的 messages；user 问题保持英文
messages = [
    {"role": "system", "content": "You are a very rude assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小的 gpt-4.1-nano 快速看 system 对风格的影响
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 展示助手回复文本
response.choices[0].message.content


In [ ]:
# ========== 语气实验：system =「友善」时，同一道 2+2 会怎样答 ==========

# 构造仅含 system + user 的 messages；user 问题保持英文
messages = [
    {"role": "system", "content": "You are a super kind and helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小的 gpt-4.1-nano 快速看 system 对风格的影响
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 展示助手回复文本
response.choices[0].message.content


In [ ]:
# ========== 语气实验：system =「意大利语风」时，同一道 2+2 会怎样答 ==========

# 构造仅含 system + user 的 messages；user 问题保持英文
messages = [
    {"role": "system", "content": "You are an italian assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小的 gpt-4.1-nano 快速看 system 对风格的影响
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 展示助手回复文本
response.choices[0].message.content


In [ ]:
# ========== 语气实验：system =「海盗腔」时，同一道 2+2 会怎样答 ==========

# 构造仅含 system + user 的 messages；user 问题保持英文
messages = [
    {"role": "system", "content": "You are a helpful assistant that talks like a pirate"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小的 gpt-4.1-nano 快速看 system 对风格的影响
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 展示助手回复文本
response.choices[0].message.content


In [ ]:
# ========== 语气实验：system =「Paisa」时，同一道 2+2 会怎样答 ==========

# 构造仅含 system + user 的 messages；user 问题保持英文
messages = [
    {"role": "system", "content": "You are a helpful assistant that talks like a colombian person from medellin, a paisa"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小的 gpt-4.1-nano 快速看 system 对风格的影响
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 展示助手回复文本
response.choices[0].message.content


## 用函数组装给摘要模型的 messages

把 system prompt、user 前缀和网页正文拼成 API 需要的列表结构。


In [ ]:
# ========== 辅助函数：website 正文 → 标准 messages 列表 ==========

def messages_for(website):
    # 返回两项：system 用全局 system_prompt；user 用前缀 + 网页正文
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# ========== 预览：user 侧实际会发送的长字符串 ==========

# 前缀 + 先前抓到的 website 正文（尚未包进 messages）
user_prompt_prefix + website


In [ ]:
# ========== 预览：messages_for(website) 的完整结构 ==========

# 可改 url 重新抓取后再看；确认 role/content 是否符合预期
messages_for(website)


## 拼起来：一次完整的「抓取 + 摘要」调用

OpenAI Chat Completions 的用法会在本课反复出现。


In [ ]:
# ========== summarize(url)：抓取网页并用 gpt-4.1-mini 生成摘要 ==========

def summarize(url):
    # 按 URL 抓取正文
    website = fetch_website_contents(url)
    # 调用 Chat Completions；messages 由 messages_for 组装
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 返回助手回复纯文本
    return response.choices[0].message.content


In [ ]:
# ========== 试跑：对前面的 url 做一次摘要 ==========

summarize(url)


In [ ]:
# ========== display_summary：摘要结果用 Markdown 漂亮展示 ==========

def display_summary(url):
    # 先拿到摘要字符串
    summary = summarize(url)
    # 在笔记本中渲染为 Markdown
    display(Markdown(summary))


In [ ]:
# ========== 展示：对默认 url 渲染 Markdown 摘要 ==========

display_summary(url)


# 再试更多网站

这种简单抓取**只适合**服务端直接返回 HTML 文本的站点。

- 大量用 JavaScript 渲染的站点（如不少 React 应用）可能抓不到正文 → 社区贡献里有 Selenium 等方案
- 受 CloudFront 等防护的站点可能 403
- 但仍有很多站点可以正常工作——多换几个 URL 试试


In [ ]:
# ========== 换站试摘要：CNN ==========

display_summary("https://cnn.com")


In [ ]:
# ========== 换站试摘要：Anthropic ==========

display_summary("https://anthropic.com")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">你刚完成了一次 Frontier Model 的 Cloud API 调用。摘要是经典 GenAI 用例：新闻、财报、简历/求职信……应用几乎无限。想想如何在自己的业务里做原型。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前 — 自己试一版</h2>
            <span style="color:#900;">用下一格做一个简单商业示例。作者提示可做「根据邮件正文建议短主题行」；本贡献者改成了「在哥伦比亚找 Data Scientist 岗位」的 LinkedIn 灵感助手（注意：模型并不能真的实时搜索 LinkedIn，输出仅供练习）。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 练习变体：用提示词模拟「哥伦比亚 Data Scientist 岗位推荐」 ==========

# 第 1 步：创建提示（拼写错误如 finde/acientist/desciption 保持原样，不改可运行字符串）
system_prompt = """
    You are a helpful assistant helping people finde their dream job in linkedin.
    You will be given a a short description of the dream job and you will need to find
    the best 3 open positions you find in linkedin for that dream job. And you have
    to focus your work for Colombia. You will return to the user a short desciption
    of the 3 positions, the name of the company, the title of the position, and the
    url of the position.
"""
user_prompt = """
    I am looking for a job as a data acientist in Colombia. Help me find the best 3 open positions for me.
"""

# 第 2 步：组装 messages 列表（system + user）
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
] # fill this in

# 第 3 步：调用 OpenAI（这里用 gpt-4o-mini）
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

# 第 4 步：打印助手回复
print(response.choices[0].message.content)


## 加分练习：更强的网页抓取

若试 `display_summary("https://openai.com")` 可能失败——该站大量依赖 JavaScript。

常见加强手段：Selenium / Playwright 等（背后起浏览器再取渲染后 DOM）。社区贡献文件夹里有同学的 Selenium 示例。有相关经验可自行增强抓取层。


# 分享你的代码

欢迎把改进提交到社区贡献文件夹（Pull Request）。作者不是 git 专家也会提供指南。

总体步骤：https://edwarddonner.com/pr

提交前检查：

1. PR 尽量只含 community-contributions 相关改动（除非另有约定）
2. 笔记本输出保持清晰
3. 总计少于约 2000 行，文件不要太多
4. 不要塞无关测试、过长 README、`.env.example`、表情符号堆砌等

详细说明示例：  
https://chatgpt.com/share/6873c22b-2a1c-8012-bc9a-debdcf7c835b

指南文件夹（Guide 3）也有完整 PR 步骤。谢谢！


In [ ]:
# （空代码格）原笔记本保留的空白单元格；无需执行
